## SQL Analysis of a Book Subscription Platform

**Project Description**

After the outbreak of the COVID-19 pandemic, everyday life changed significantly. People started spending less time outside and more time online. Interest in reading increased, as books replaced activities such as visiting cafes and shopping centers.

Observing this trend, our company acquired a popular book subscription service. As analysts, our task is to use SQL analysis to understand the structure of the service’s database and identify opportunities for product development.

---

**Project Objective**

Based on the book service database:

- Identify the activity of publishers, authors, and readers  
- Determine popular and highly rated books  
- Identify active users  
- Provide insights to improve the product  

---

**Data Description**

- **Table: books** — contains information about books:
  - book_id — book ID  
  - author_id — author ID  
  - title — book title  
  - num_pages — number of pages  
  - publication_date — publication date  
  - publisher_id — publisher ID  

- **Table: authors** — contains information about authors:
  - author_id — author ID  
  - author — author name  

- **Table: publishers** — contains information about publishers:
  - publisher_id — publisher ID  
  - publisher — publisher name  

- **Table: ratings** — contains user ratings of books:
  - rating_id — rating ID  
  - book_id — book ID  
  - username — username of the reviewer  
  - rating — book rating  

- **Table: reviews** — contains user reviews:
  - review_id — review ID  
  - book_id — book ID  
  - username — reviewer username  
  - text — review text  

---

**Project Plan**

- [Database access and data exploration](#data_section1)  
- [Task solutions](#data_section2)  
- [Final insights and recommendations](#data_section3)  

<a id="data_section1"></a>
## Доступ к базе данных и изучение данных

#### Импорт библиотек 

In [1]:
# импортируем библиотеки
import pandas as pd
from sqlalchemy import text, create_engine
import sqlalchemy as sa
from IPython.display import display

#### Установка параметров для работы с sql

In [2]:
# параметры подключения
db_config = {
    'user': 'praktikum_student',
    'pwd': 'Sdf4$2;d-d30pp',
    'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
    'port': 6432,
    'db': 'data-analyst-final-project-db'
}

connection_string = 'postgresql://{user}:{pwd}@{host}:{port}/{db}'.format(**db_config)
engine = create_engine(connection_string, connect_args={'sslmode':'require'})

# функция
def get_sql_data(query: str, engine=engine) -> pd.DataFrame:
    with engine.connect() as con:
        return pd.read_sql(text(query), con)


#### Выведем количество строк и первые строки  из каждой таблицы

In [3]:
# список таблиц
tables = ['books', 'authors', 'publishers', 'ratings', 'reviews']

# перебираем таблицы
for t in tables:
    print(f"Таблица: {t}")
    
    # вывод количества строк
    count_query = f"SELECT COUNT(*) AS row_count FROM {t}"
    count = get_sql_data(count_query).iloc[0, 0]
    print(f"Количество строк: {count}")
    
    # вывод первых 5 строк
    select_query = f"SELECT * FROM {t} LIMIT 5"
    display(get_sql_data(select_query))


Таблица: books
Количество строк: 1000


,book_id,author_id,title,num_pages,publication_date,publisher_id
0,1,546,'Salem's Lot,594,2005-11-01,93
1,2,465,1 000 Places to See Before You Die,992,2003-05-22,336
2,3,407,13 Little Blue Envelopes (Little Blue Envelope...,322,2010-12-21,135
3,4,82,1491: New Revelations of the Americas Before C...,541,2006-10-10,309
4,5,125,1776,386,2006-07-04,268


Таблица: authors
Количество строк: 636


,author_id,author
0,1,A.S. Byatt
1,2,Aesop/Laura Harris/Laura Gibbs
2,3,Agatha Christie
3,4,Alan Brennert
4,5,Alan Moore/David Lloyd


Таблица: publishers
Количество строк: 340


,publisher_id,publisher
0,1,Ace
1,2,Ace Book
2,3,Ace Books
3,4,Ace Hardcover
4,5,Addison Wesley Publishing Company


Таблица: ratings
Количество строк: 6456


,rating_id,book_id,username,rating
0,1,1,ryanfranco,4
1,2,1,grantpatricia,2
2,3,1,brandtandrea,5
3,4,2,lorichen,3
4,5,2,mariokeller,2


Таблица: reviews
Количество строк: 2793


,review_id,book_id,username,text
0,1,1,brandtandrea,Mention society tell send professor analysis. ...
1,2,1,ryanfranco,Foot glass pretty audience hit themselves. Amo...
2,3,2,lorichen,Listen treat keep worry. Miss husband tax but ...
3,4,3,johnsonamanda,Finally month interesting blue could nature cu...
4,5,3,scotttamara,Nation purpose heavy give wait song will. List...


**ВЫВОД**
- В базе содержится 5 таблиц

- В базе 1000 книг, 636 авторов, 340 издательств

- Пользователи оставили 6456 оценок и 2793 обзора

- Таблицы хорошо связаны по book_id, author_id, publisher_id, username

- Структура данных полная, охватывает как числовые, так и текстовые признаки

- Объёма и качества данных достаточно для аналитики

<a id="data_section2"></a>
## Решение заданий

#### Посчитайте, сколько книг вышло после 1 января 2000 года

In [4]:
query_1 = '''
SELECT COUNT(*) AS books_after_2000
FROM books
WHERE publication_date > '2000-01-01'
'''

get_sql_data(query_1)

,books_after_2000
0,819


**Выводы**

- В базе 819 книг было опубликовано после 1 января 2000 года.
- Это говорит о том, что основная часть каталога — современные издания, что хорошо для актуального читательского интереса и формирования рекомендаций.

#### Для каждой книги посчитайте количество обзоров и среднюю оценку

In [5]:
query_2 = '''
SELECT b.book_id,
    b.title,
    COUNT(DISTINCT rv.review_id) AS review_count,
    ROUND(AVG(rt.rating), 2) AS avg_rating
FROM books b
LEFT JOIN reviews rv ON b.book_id = rv.book_id
LEFT JOIN ratings rt ON b.book_id = rt.book_id
GROUP BY b.book_id, b.title
ORDER BY avg_rating DESC
'''

get_sql_data(query_2)

,book_id,title,review_count,avg_rating
0,86,Arrows of the Queen (Heralds of Valdemar #1),2,5.00
1,901,The Walking Dead Book One (The Walking Dead #...,2,5.00
2,390,Light in August,2,5.00
3,972,Wherever You Go There You Are: Mindfulness Me...,2,5.00
4,136,Captivating: Unveiling the Mystery of a Woman'...,2,5.00
...,...,...,...,...
995,915,The World Is Flat: A Brief History of the Twen...,3,2.25
996,316,His Excellency: George Washington,2,2.00
997,202,Drowning Ruth,3,2.00
998,371,Junky,2,2.00


**ВЫВОДЫ:**
- Несколько книг получили идеальную оценку 5.00, но только при 2 отзывах — например:
"The Walking Dead Book One", "Light in August", "Wherever You Go There You Are". Рейтинг может быть завышен из-за недостаточного количества отзывов.
- Внизу списка — книги с низкими рейтингами (2.00 и ниже) при таком же малом числе отзывов, например: "Junky", "Harvesting the Heart".
- Средние рейтинги при малом количестве отзывов (1–3) не надёжны для выводов о качестве книги.

#### Определите издательство, которое выпустило наибольшее число книг толще 50 страниц — так вы исключите из анализа брошюры

In [6]:
query_3 = '''
SELECT 
    p.publisher,
    COUNT(b.book_id) AS book_count
FROM books b
JOIN publishers p ON b.publisher_id = p.publisher_id
WHERE b.num_pages > 50
GROUP BY p.publisher
ORDER BY book_count DESC
LIMIT 1
'''

get_sql_data(query_3)


,publisher,book_count
0,Penguin Books,42


**ВЫВОДЫ:**
- Издательство Penguin Books выпустило наибольшее число книг толщиной более 50 страниц — всего 42.
- Это говорит о том, что Penguin Books — один из самых продуктивных и надёжных партнёров, чьи издания соответствуют полноценному книжному формату (а не брошюрам).

#### Определите автора с самой высокой средней оценкой книг — учитывайте только книги с 50 и более оценками

In [7]:
query_4= '''
WITH book_stats AS (
    SELECT 
        b.book_id,
        b.author_id,
        COUNT(rt.rating) AS rating_count,
        AVG(rt.rating) AS avg_rating
    FROM books b
    JOIN ratings rt ON b.book_id = rt.book_id
    GROUP BY b.book_id
    HAVING COUNT(rt.rating) >= 50
),
author_stats AS (
    SELECT 
        bs.author_id,
        ROUND(AVG(bs.avg_rating), 2) AS avg_author_rating,
        SUM(bs.rating_count) AS total_ratings
    FROM book_stats bs
    GROUP BY bs.author_id
)
SELECT 
    a.author,
    ast.avg_author_rating,
    ast.total_ratings
FROM author_stats ast
JOIN authors a ON ast.author_id = a.author_id
ORDER BY ast.avg_author_rating DESC
LIMIT 1;

'''

get_sql_data(query_4)


,author,avg_author_rating,total_ratings
0,J.K. Rowling/Mary GrandPré,4.28,310.0


**ВЫВОДЫ:**
- Автор J.K. Rowling/Mary GrandPré получил в сумме 310 оценок на свои книги.

- Средняя оценка составила 4.28, что является одной из самых высоких среди всех авторов с большим количеством голосов.

- Это говорит о высоком и устойчивом интересе аудитории к книгам этого автора (авторского дуэта).

#### Посчитайте среднее количество обзоров от пользователей, которые поставили больше 48 оценок

In [8]:
query_5 = '''
WITH active_users AS (
    SELECT username
    FROM ratings
    GROUP BY username
    HAVING COUNT(rating) > 48
)

SELECT 
    ROUND(AVG(review_count), 2) AS avg_reviews
FROM (
    SELECT username, COUNT(*) AS review_count
    FROM reviews
    WHERE username IN (SELECT username FROM active_users)
    GROUP BY username
) AS sub
'''

get_sql_data(query_5)


,avg_reviews
0,24.0


**ВЫВОДЫ:**
- Среди активных пользователей (тех, кто поставил более 48 оценок), в среднем каждый оставил 24 обзора.

- Это означает, что вовлечённые пользователи не только оценивают книги, но и активно делятся своим мнением в виде обзоров.

- Такие пользователи — ценная аудитория для развития сервиса: их можно поощрять, вовлекать в тестирование новых функций, давать бонусы за активность.

<a id="data_section3"></a>
## Final Insights and Recommendations

**CONCLUSIONS:** 

- The platform’s catalog contains a large number of modern publications — 819 out of 1,000 books were published after January 1, 2000. This makes the dataset relevant and appealing to users.

- Book quality and popularity vary: some books have high ratings (4.5–5.0) but very few reviews — such cases should be filtered. Most books have 2–4 reviews, indicating moderate user activity.

- The most productive publisher is Penguin Books, which has published 42 books with more than 50 pages. This makes it a valuable partner for the platform.

- J.K. Rowling / Mary GrandPré is the most popular author duo in the dataset: 310 ratings and a high average rating of 4.28 indicate strong and consistent reader interest.

- Active users not only rate books but also write reviews — users who gave more than 48 ratings write an average of 24 reviews. This represents a highly engaged audience.

---

**RECOMMENDATIONS:** 

- Promote popular authors and major publishers  

- Consider only books with a sufficient number of ratings  

- Engage active users through gamification and incentives  

- Use this data to improve personalized recommendations  